# DeepSeek agent + MCP (LangChain, traced with LangSmith)

This notebook connects a **DeepSeek agent (built with LangChain)** to the local MCP SQLite server,
so the agent can query the sample database through the server's tools.

Every run is traced to **LangSmith** (project `dsr_batch_47`), driven by the credentials in the
`.env` file next to this notebook.

## Before running
1. Start the MCP server in a terminal, from the `mcp_sqlite_example` folder:
   ```bash
   python mcp_server.py
   ```
   It prints `Serving MCP over Streamable HTTP at http://127.0.0.1:8000/mcp`.
2. Make sure `.env` exists in this folder (it holds `DEEPSEEK_API_KEY` and the `LANGSMITH_*` values).

## How it works
- `.env` is loaded first so LangSmith tracing + the DeepSeek key are set before the agent runs.
- `langchain-mcp-adapters` loads the server's MCP tools over Streamable HTTP.
- A DeepSeek chat model (`deepseek:deepseek-v4-flash`) powers a LangChain `create_agent`.
- The agent chooses which tool to call (e.g. `run_query`) and answers in plain English.

In [1]:
# Install once if these aren't available yet:
# %pip install langchain langchain-deepseek langchain-mcp-adapters python-dotenv

In [2]:
import os
import getpass
from pathlib import Path

from dotenv import load_dotenv

# --- Load credentials BEFORE importing/building LangChain objects ---
# This makes sure LANGSMITH_* and DEEPSEEK_API_KEY are set so runs are traced.
def find_env():
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        env = base / ".env"
        if env.is_file():
            return env
    return None

env_path = find_env()
if env_path is None:
    fallback = Path.cwd() / "mcp_sqlite_example" / ".env"
    if fallback.is_file():
        env_path = fallback

if env_path is not None:
    load_dotenv(env_path, override=True)
    print(f"Loaded environment from {env_path}")
else:
    print("No .env file found; relying on pre-existing environment variables.")

# Fallback: prompt for the DeepSeek key if it still isn't set.
if not os.environ.get("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("DeepSeek API key: ")

print("LangSmith tracing:", os.environ.get("LANGSMITH_TRACING"),
      "| project:", os.environ.get("LANGSMITH_PROJECT"))
print("DeepSeek key:", "set" if os.environ.get("DEEPSEEK_API_KEY") else "MISSING")

MCP_URL = "http://127.0.0.1:8000/mcp"

Loaded environment from /Users/davidjeon/Documents/dsr_materials/agentic_ai/mcp_sqlite_example/.env
LangSmith tracing: true | project: dsr_batch_47
DeepSeek key: set


In [3]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect LangChain to the running MCP server and load its tools.
client = MultiServerMCPClient({
    "sqlite": {"transport": "streamable_http", "url": MCP_URL},
})

async def load_tools():
    tools = await client.get_tools()
    print(f"Loaded {len(tools)} tools from the MCP server:")
    for t in tools:
        print(f"  - {t.name}: {t.description}")
    return tools

tools = await load_tools()

Loaded 3 tools from the MCP server:
  - list_tables: Return the names of all tables available in the sample database.
  - table_schema: Return the columns (and their types) of a given table.

    Args:
        table_name: Name of the table to describe, e.g. "employees".
    
  - run_query: Execute a single read-only SELECT query and return the resulting rows as JSON.

    Args:
        sql: A SELECT statement, e.g. "SELECT * FROM employees WHERE salary > 90000".
    


In [4]:
# Build the DeepSeek agent with the MCP tools.
model = init_chat_model("deepseek:deepseek-v4-flash")

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant with access to a sample company database. "
        "Use the provided tools to inspect the schema and run read-only queries. "
        "Answer the user's question in plain English, citing the data you found."
    ),
)
print("Agent ready:", type(agent).__name__)

Agent ready: CompiledStateGraph


In [5]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage

async def ask(question: str):
    return await agent.ainvoke({"messages": [{"role": "user", "content": question}]})

def show(result):
    for m in result["messages"]:
        if isinstance(m, HumanMessage):
            print(f"\n[user] {m.content}")
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"[agent -> {tc['name']}] {tc['args']}")
            else:
                print(f"\n[assistant] {m.content}")
        elif isinstance(m, ToolMessage):
            print(f"[tool result] {str(m.content)[:200]}")

result = await ask(
    "Which employees earn more than 90,000? List their names, departments "
    "and salaries, highest first."
)
show(result)


[user] Which employees earn more than 90,000? List their names, departments and salaries, highest first.
[agent -> list_tables] {}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "customers"\n  },\n  {\n    "name": "employees"\n  },\n  {\n    "name": "products"\n  }\n]', 'id': 'lc_4e1911b4-40bf-4da1-924e-fd2fb49c982a'}]
[agent -> table_schema] {'table_name': 'employees'}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "id",\n    "type": "INTEGER",\n    "notnull": false\n  },\n  {\n    "name": "name",\n    "type": "TEXT",\n    "notnull": true\n  },\n  {\n    "name": "de
[agent -> run_query] {'sql': 'SELECT name, department, salary FROM employees WHERE salary > 90000 ORDER BY salary DESC'}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "Frank Brown",\n    "department": "Engineering",\n    "salary": 101000.0\n  },\n  {\n    "name": "Grace Hopper",\n    "department": "Data",\n    "salary"

[assistant] Here are the employees earning more than $90,000

## Ask your own question

Edit the question below and run the cell. Each run creates a new trace in LangSmith
(project `dsr_batch_47`) showing the model calls and MCP tool invocations.

In [6]:
result = await ask(
    "How many products are low in stock (under 50 units), and what are they?"
)
show(result)


[user] How many products are low in stock (under 50 units), and what are they?
[agent -> list_tables] {}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "customers"\n  },\n  {\n    "name": "employees"\n  },\n  {\n    "name": "products"\n  }\n]', 'id': 'lc_a22047b9-61e2-433d-97c4-b806315d0432'}]
[agent -> table_schema] {'table_name': 'products'}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "id",\n    "type": "INTEGER",\n    "notnull": false\n  },\n  {\n    "name": "name",\n    "type": "TEXT",\n    "notnull": true\n  },\n  {\n    "name": "ca
[agent -> run_query] {'sql': 'SELECT name, category, price, stock FROM products WHERE stock < 50 ORDER BY stock ASC;'}
[tool result] [{'type': 'text', 'text': '[\n  {\n    "name": "Standing Desk",\n    "category": "Furniture",\n    "price": 399.0,\n    "stock": 12\n  },\n  {\n    "name": "Desk Chair",\n    "category": "Furniture",\

[assistant] There are **4 products** low in stock (under 50 units):

| Product | Category